In [1]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset
from transformers import T5Tokenizer, T5ForConditionalGeneration
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

In [ ]:
## LOAD DATA

class Data(Dataset):
    def __init__(self, examples):
        self.examples = examples

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        item = self.examples.iloc[idx]
        label = "yes" if item["label"] == 1 else "no"
        input = item['input']
        return input, label

train_data_df = pd.read_csv('/path/to/T_paraphrase.csv') # load training dataset. load T_standard for TS0 and TS1
eval_data_df = pd.read_csv('/path/to/V_standard.csv')
eval_para_data_df = pd.read_csv('/path/to/V_paraphrase.csv')

# randomly sample 2048 examples from each of the datasets
train_subset_df = train_data_df.sample(n=2048).reset_index(drop=True)
eval_std_subset_df = eval_data_df.sample(n=2048).reset_index(drop=True)
eval_para_subset_df = eval_para_data_df.sample(n=2048).reset_index(drop=True)

# apply DataLoaders to each subset for batching
train_subset_loader = DataLoader(Data(train_subset_df), batch_size=128)
eval_std_subset_loader = DataLoader(Data(eval_std_subset_df), batch_size=128)
eval_para_subset_loader = DataLoader(Data(eval_para_subset_df), batch_size=128)

In [ ]:
## LOAD MODEL

model_name = "google/flan-t5-large"
tokeniser = T5Tokenizer.from_pretrained(model_name)
model_loc = "path/to/model/weights" 
model = T5ForConditionalGeneration.from_pretrained(model_loc,
                                                   torch_dtype=torch.bfloat16,
                                                   device_map='cuda')

In [4]:
def compute_vectors(data_loader, model, tokeniser):
    model.eval()
    vectors = []
    labels = []
    with torch.no_grad():
        for batch in data_loader: # iterate over batches
            texts, batch_labels = batch
            inputs = tokeniser(texts, return_tensors="pt", truncation=True, padding=True, max_length=512).to("cuda") # tokenise inputs

            # obtain final decoder representations
            batch_size = inputs.input_ids.shape[0]
            decoder_input = torch.full((batch_size, 1), model.config.decoder_start_token_id, dtype=torch.long, device='cuda')
            outputs = model(input_ids=inputs.input_ids, decoder_input_ids=decoder_input, output_hidden_states=True)
            decoder_output = outputs.decoder_hidden_states[-1][:, -1, :]

            vectors.append(decoder_output)
            labels.extend(batch_labels)

    vectors = torch.cat(vectors, dim=0)
    return vectors.to(torch.float32).cpu().numpy(), labels

In [ ]:
def plot_vectors(vectors, labels, save_path, epoch, w_yes, w_no):
    binary_labels = np.array([1 if l.lower() == 'yes' else 0 for l in labels]) # convert labels to binary

    # perform PCA with 2 components on vectors
    pca = PCA(n_components=2) 
    X = pca.fit_transform(vectors)

    true_boundary = (w_yes - w_no).numpy() # retrieve decision boundary for 'yes' and 'no' logits
    pca_boundary = pca.components_ @ true_boundary # project to 2D subspace formed by PCA

    # estimated accuracy metrics
    preds = X @ pca_boundary # obtain predictions from projected data and decision boundary
    predictions = np.array([1 if pred >=0 else 0 for pred in preds]) # convert predictions to binary
    accuracy = np.mean(predictions == binary_labels) # obtain accuracy

    # configure grid for plotting decision boundary
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200), np.linspace(y_min, y_max, 200))
    grid_points = np.c_[xx.ravel(), yy.ravel()]

    # split the plot by decision boundary, used to colour either side of it
    grid_decision_scores = grid_points @ pca_boundary
    Z = (grid_decision_scores >= 0).astype(int).reshape(xx.shape)

    plt.figure(figsize=(8, 6))
    for label in np.unique(binary_labels): # plot projected examples
        plt.scatter(X[(binary_labels == label), 0], X[(binary_labels == label), 1], s=5, label='Causal Examples' if label == 1 else 'Non-Causal Examples', alpha=0.7)

    plt.contourf(xx, yy, Z, alpha=0.3, cmap=plt.cm.coolwarm) # plot decision boundary, coloured with the positive and negative side

    # plot the epoch, overall accuracy, and PCA variance metrics
    ax = plt.gca()
    ax.text(0.02, 0.98, f"Epoch: {epoch}",
            transform=ax.transAxes, fontsize=12, verticalalignment='top',
            bbox=dict(facecolor='white', alpha=0.6))

    ax.text(0.02, 0.90, f"Accuracy: {accuracy:.3f}",
            transform=ax.transAxes, fontsize=12, verticalalignment='top',
            bbox=dict(facecolor='white', alpha=0.6))

    total_var_explained = pca.explained_variance_ratio_.sum() * 100
    ax.text(0.02, 0.82, f"PCA Var. Expl.: {total_var_explained:.2f}%",
            transform=ax.transAxes, fontsize=12, verticalalignment='top',
            bbox=dict(facecolor='white', alpha=0.6))

    plt.legend()
    plt.savefig(save_path)
    plt.close()

In [ ]:
w_yes = model.lm_head.weight[4273].detach().cpu().to(torch.float32) # get weight row for 'yes' logit
w_no = model.lm_head.weight[150].detach().cpu().to(torch.float32) # get weight row for 'no' logit

# compute vectors for all datasets
train_vectors, train_labels = compute_vectors(train_subset_loader, model, tokeniser)
eval_vectors, eval_labels = compute_vectors(eval_std_subset_loader, model, tokeniser)
para_vectors, para_labels = compute_vectors(eval_para_subset_loader, model, tokeniser)

# generate PCA plots
plot_vectors(train_vectors, train_labels, 'frame_1.png', 10, w_yes, w_no)
plot_vectors(eval_vectors, eval_labels, 'frame_2.png', 10, w_yes, w_no)
plot_vectors(para_vectors, para_labels, 'frame_3.png', 10, w_yes, w_no)